# 🏠 Airbnb Open Data — Automatic Data Cleaning Pipeline
**Upload your `Airbnb_Open_Data.csv` and run all cells top to bottom.**  
The cleaned file `cleaned_airbnb.csv` will be saved and downloaded automatically.

---
### Pipeline Steps
| # | Step |
|---|------|
| 1 | Imports & Setup |
| 2 | File Upload |
| 3 | Column Configuration |
| 4 | Data Inspection |
| 5 | Fix Data Types |
| 6 | Remove Duplicates |
| 7 | Handle Missing Values |
| 8 | Detect Outliers |
| 9 | Standardise Text |
| 10 | Before vs After Visualisations 📊 |
| 11 | Cleaning Report |
| 12 | Save & Download |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

print("✅ Libraries loaded successfully.")


In [ ]:
from google.colab import files

print("📂 Please upload your Airbnb_Open_Data.csv file...")
uploaded = files.upload()

FILE_NAME = list(uploaded.keys())[0]
print(f"✅ Uploaded: {FILE_NAME}")


In [ ]:
# ── Column groups for the Airbnb Open Data dataset ─────────────────────────
DATE_COLS    = ["last review"]

NUMERIC_COLS = ["price", "service fee", "minimum nights",
                "number of reviews", "reviews per month",
                "review rate number", "calculated host listings count",
                "availability 365", "Construction year", "lat", "long"]

TEXT_COLS    = ["NAME", "host name", "neighbourhood group",
                "neighbourhood", "country", "country code",
                "cancellation_policy", "room type",
                "host_identity_verified", "house_rules"]

# Columns used for before/after visualisations
VIZ_COL_1 = "price"
VIZ_COL_2 = "number of reviews"

print("✅ Column configuration ready.")


In [ ]:
def inspect_data(df, label="Dataset"):
    print(f"\n{'='*60}")
    print(f"  📋  {label}")
    print(f"{'='*60}")
    print(f"  Shape         : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Duplicate rows: {df.duplicated().sum():,}")
    print(f"\n── Data Types ──────────────────────────────────────────")
    print(df.dtypes.to_string())
    print(f"\n── Missing Values ──────────────────────────────────────")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    miss_df = pd.DataFrame({"Missing": missing, "% Missing": missing_pct})
    print(miss_df[miss_df["Missing"] > 0].to_string())
    print(f"\n── Numeric Summary ─────────────────────────────────────")
    print(df.describe(include=[np.number]).T.to_string())
    print(f"\n── Sample Categorical Value Counts ─────────────────────")
    for col in ["neighbourhood group", "room type", "cancellation_policy"]:
        if col in df.columns:
            print(f"\n  {col}:")
            print(df[col].value_counts().head(5).to_string())
    print(f"{'='*60}\n")


In [ ]:
def fix_types(df, date_cols=DATE_COLS, numeric_cols=NUMERIC_COLS):
    df = df.copy()
    print("\n🔧 Fixing data types...")

    for col in date_cols:
        if col in df.columns:
            before = df[col].dtype
            df[col] = pd.to_datetime(df[col], errors="coerce")
            print(f"  📅 '{col}': {before} → datetime64")

    for col in numeric_cols:
        if col in df.columns:
            before = df[col].dtype
            if df[col].dtype == object:
                df[col] = (df[col].astype(str)
                           .str.replace(r"[\$,\s]", "", regex=True)
                           .replace({"nan": np.nan, "": np.nan}))
            df[col] = pd.to_numeric(df[col], errors="coerce")
            if str(before) != str(df[col].dtype):
                print(f"  🔢 '{col}': {before} → {df[col].dtype}")

    print("  ✅ Type fixing complete.")
    return df


In [ ]:
def remove_duplicates(df):
    n = df.duplicated().sum()
    if n == 0:
        print("  ✅ No duplicate rows found.")
        return df
    df = df.drop_duplicates()
    print(f"  🗑️  Removed {n:,} duplicate rows. Remaining: {len(df):,}")
    return df


In [ ]:
def handle_missing(df, strategy="smart"):
    """
    strategy:
      'smart'  – median if |skew| > 1, else mean; mode for categoricals
      'mean'   – mean for numerics, mode for categoricals
      'median' – median for numerics, mode for categoricals
      'mode'   – mode for everything
      'drop'   – drop all rows with any missing value
    """
    df = df.copy()
    total = df.isnull().sum().sum()
    if total == 0:
        print("  ✅ No missing values.")
        return df

    print(f"  ⚠️  Missing cells before: {total:,}")

    if strategy == "drop":
        before = len(df)
        df = df.dropna()
        print(f"  🗑️  Dropped {before - len(df):,} rows.")
        return df

    for col in df.select_dtypes(include=[np.number]).columns:
        n_miss = df[col].isnull().sum()
        if n_miss == 0:
            continue
        if strategy == "mean":
            fv, label = df[col].mean(), "mean"
        elif strategy == "median":
            fv, label = df[col].median(), "median"
        elif strategy == "smart":
            sk = df[col].skew()
            if abs(sk) > 1:
                fv, label = df[col].median(), f"median (skew={sk:.2f})"
            else:
                fv, label = df[col].mean(), f"mean (skew={sk:.2f})"
        else:
            fv = df[col].mode()[0] if not df[col].mode().empty else 0
            label = "mode"
        df[col] = df[col].fillna(fv)
        print(f"  🔧 '{col}': filled {n_miss:,} nulls with {label} ({fv:.4f})")

    for col in df.select_dtypes(include=["object", "bool"]).columns:
        n_miss = df[col].isnull().sum()
        if n_miss == 0:
            continue
        fv = df[col].mode()[0] if not df[col].mode().empty else "Unknown"
        df[col] = df[col].fillna(fv)
        print(f"  🔧 '{col}': filled {n_miss:,} nulls with mode ('{fv}')")

    print(f"  ✅ Missing cells after: {df.isnull().sum().sum():,}")
    return df


In [ ]:
def detect_outliers(df, method="iqr", z_thresh=3.0, remove=False):
    if method == "none":
        print("  ⏭️  Outlier detection skipped.")
        return df

    df = df.copy()
    num_df = df.select_dtypes(include=[np.number])

    if method == "iqr":
        Q1, Q3 = num_df.quantile(0.25), num_df.quantile(0.75)
        IQR = Q3 - Q1
        mask = ((num_df < (Q1 - 1.5 * IQR)) |
                (num_df > (Q3 + 1.5 * IQR))).any(axis=1)
    elif method == "zscore":
        from scipy import stats
        z = np.abs(stats.zscore(num_df, nan_policy="omit"))
        mask = (z > z_thresh).any(axis=1)
    else:
        print("  ⚠️  Unknown method. Skipping.")
        return df

    n = mask.sum()
    print(f"  🔍 {n:,} outlier rows detected via {method.upper()} "
          f"({n/len(df)*100:.2f}% of data).")

    if remove and n > 0:
        df = df[~mask]
        print(f"  🗑️  Removed outliers. Remaining: {len(df):,} rows.")

    return df


In [ ]:
def standardize_text(df, text_cols=TEXT_COLS):
    df = df.copy()
    for col in text_cols:
        if col not in df.columns or df[col].dtype != object:
            continue
        df[col] = (df[col].astype(str)
                   .str.lower()
                   .str.strip()
                   .str.replace(r"[^a-z0-9 ,\-\']", " ", regex=True)
                   .str.replace(r"\s{2,}", " ", regex=True)
                   .str.strip()
                   .replace("nan", np.nan))
        print(f"  📝 Standardised: '{col}'")
    return df


## 📊 Before vs After Visualisations
Histograms and boxplots for `price` and `number of reviews`.

In [ ]:
def plot_before_after(original_df, cleaned_df,
                      col1=VIZ_COL_1, col2=VIZ_COL_2):

    def _to_num(series):
        return pd.to_numeric(
            series.astype(str).str.replace(r"[\$,\s]", "", regex=True),
            errors="coerce").dropna()

    cols = [c for c in [col1, col2]
            if c in original_df.columns and c in cleaned_df.columns]
    if not cols:
        print("Columns not found for visualisation.")
        return

    BEFORE_COLOR = "#B0B0B0"
    AFTER_COLORS = ["#E07B54", "#4A90D9"]

    fig = plt.figure(figsize=(16, 12))
    fig.suptitle("Before vs After Cleaning", fontsize=18,
                 fontweight="bold", y=1.01)
    gs = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)

    for i, col in enumerate(cols):
        before = _to_num(original_df[col])
        after  = _to_num(cleaned_df[col])

        # ── Histogram ───────────────────────────────────────────────────────
        ax_hist = fig.add_subplot(gs[0, i])
        cap99   = after.quantile(0.99)
        ax_hist.hist(before.clip(upper=cap99), bins=60,
                     alpha=0.5, color=BEFORE_COLOR,
                     label=f"Before (n={len(before):,})", density=True)
        ax_hist.hist(after.clip(upper=cap99), bins=60,
                     alpha=0.7, color=AFTER_COLORS[i],
                     label=f"After  (n={len(after):,})", density=True)
        ax_hist.set_title(f"Distribution — {col}", fontsize=13,
                          fontweight="bold")
        ax_hist.set_xlabel(col, fontsize=11)
        ax_hist.set_ylabel("Density", fontsize=11)
        ax_hist.legend(fontsize=10)

        # ── Boxplot ──────────────────────────────────────────────────────────
        ax_box = fig.add_subplot(gs[1, i])
        bp = ax_box.boxplot(
            [before.clip(upper=cap99), after.clip(upper=cap99)],
            labels=["Before", "After"],
            patch_artist=True,
            widths=0.5,
            medianprops=dict(color="black", linewidth=2.5),
        )
        bp["boxes"][0].set(facecolor=BEFORE_COLOR, alpha=0.7)
        bp["boxes"][1].set(facecolor=AFTER_COLORS[i], alpha=0.7)
        ax_box.set_title(f"Boxplot — {col}", fontsize=13, fontweight="bold")
        ax_box.set_ylabel(col, fontsize=11)

    plt.tight_layout()
    plt.savefig("before_after_cleaning.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("📈 Visualisations saved as 'before_after_cleaning.png'")


# ── Comparison stats table ───────────────────────────────────────────────────
def print_stats_comparison(original_df, cleaned_df,
                            cols=None):
    if cols is None:
        cols = [VIZ_COL_1, VIZ_COL_2]

    def _to_num(s):
        return pd.to_numeric(
            s.astype(str).str.replace(r"[\$,\s]", "", regex=True),
            errors="coerce")

    print(f"\n{'='*62}")
    print(f"  {'Column / Stat':<28} {'Before':>14} {'After':>14}")
    print(f"  {'-'*58}")
    for col in cols:
        if col not in original_df.columns or col not in cleaned_df.columns:
            continue
        b = _to_num(original_df[col]).dropna()
        a = _to_num(cleaned_df[col]).dropna()
        Q1b, Q3b = b.quantile(0.25), b.quantile(0.75)
        Q1a, Q3a = a.quantile(0.25), a.quantile(0.75)
        iqr_mask_b = ((b < Q1b - 1.5*(Q3b-Q1b)) | (b > Q3b + 1.5*(Q3b-Q1b))).sum()
        iqr_mask_a = ((a < Q1a - 1.5*(Q3a-Q1a)) | (a > Q3a + 1.5*(Q3a-Q1a))).sum()
        for stat, bv, av in [
            ("mean",     b.mean(),   a.mean()),
            ("median",   b.median(), a.median()),
            ("std",      b.std(),    a.std()),
            ("outliers", iqr_mask_b, iqr_mask_a),
        ]:
            label = f"{col} / {stat}"
            print(f"  {label:<30} {bv:>13,.2f} {av:>13,.2f}")
        print(f"  {'-'*58}")
    print(f"{'='*62}\n")


## 📋 Cleaning Report

In [ ]:
def generate_cleaning_report(original_df, cleaned_df):
    print("\n" + "="*60)
    print("  📊  CLEANING REPORT")
    print("="*60)

    print(f"\n  {'Metric':<35} {'Before':>10} {'After':>10}")
    print(f"  {'-'*55}")
    print(f"  {'Rows':<35} {original_df.shape[0]:>10,} {cleaned_df.shape[0]:>10,}")
    print(f"  {'Columns':<35} {original_df.shape[1]:>10} {cleaned_df.shape[1]:>10}")
    print(f"  {'Duplicate rows':<35} {original_df.duplicated().sum():>10,} {cleaned_df.duplicated().sum():>10,}")
    print(f"  {'Total missing cells':<35} {original_df.isnull().sum().sum():>10,} {cleaned_df.isnull().sum().sum():>10,}")

    print(f"\n  {'Column-level Missing Changes':}")
    print(f"  {'-'*55}")
    for col in original_df.columns:
        bm = original_df[col].isnull().sum()
        am = cleaned_df[col].isnull().sum() if col in cleaned_df else 0
        if bm > 0 or am > 0:
            icon = "✅" if am == 0 else "⚠️ "
            print(f"  {icon} {col:<38} {bm:>6,} → {am:>6,}")

    print(f"\n  {'Data Type Changes':}")
    print(f"  {'-'*55}")
    changed = False
    for col in original_df.columns:
        b = str(original_df[col].dtype)
        a = str(cleaned_df[col].dtype) if col in cleaned_df else "—"
        if b != a:
            print(f"  🔄 {col:<38} {b:>10} → {a}")
            changed = True
    if not changed:
        print("  (no type changes)")

    print("\n" + "="*60 + "\n")


## 🚀 Run the Full Cleaning Pipeline

In [ ]:
# ── Load raw data ────────────────────────────────────────────────────────────
print("📥 Loading raw data...")
df_raw = pd.read_csv(FILE_NAME, low_memory=False)
original_df = df_raw.copy()
print(f"Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

# ── Inspect raw data ─────────────────────────────────────────────────────────
inspect_data(df_raw, label="RAW DATA")

# ── Step 1: Fix types ────────────────────────────────────────────────────────
print("\n" + "─"*50)
df = fix_types(df_raw)

# ── Step 2: Remove duplicates ────────────────────────────────────────────────
print("\n" + "─"*50)
print("\n🔁 Removing duplicates...")
df = remove_duplicates(df)

# ── Step 3: Handle missing values ────────────────────────────────────────────
print("\n" + "─"*50)
print("\n🧹 Handling missing values (strategy = smart)...")
df = handle_missing(df, strategy="smart")

# ── Step 4: Detect outliers (no removal by default) ──────────────────────────
print("\n" + "─"*50)
print("\n📊 Detecting outliers...")
df = detect_outliers(df, method="iqr", remove=False)

# ── Step 5: Standardise text ─────────────────────────────────────────────────
print("\n" + "─"*50)
print("\n📝 Standardising text columns...")
df = standardize_text(df)

cleaned_df = df.copy()
print("\n✅ Cleaning complete!")


In [ ]:
# ── Before vs After Visualisations ───────────────────────────────────────────
plot_before_after(original_df, cleaned_df)

# ── Stats Comparison Table ────────────────────────────────────────────────────
print_stats_comparison(original_df, cleaned_df)


In [ ]:
generate_cleaning_report(original_df, cleaned_df)


## 💾 Save & Download Cleaned CSV

In [ ]:
OUTPUT_FILE = "cleaned_airbnb.csv"
cleaned_df.to_csv(OUTPUT_FILE, index=False)
print(f"💾 Saved: '{OUTPUT_FILE}'  |  "
      f"Shape: {cleaned_df.shape[0]:,} rows × {cleaned_df.shape[1]} columns")

from google.colab import files
files.download(OUTPUT_FILE)
files.download("before_after_cleaning.png")
print("📥 Downloads started!")
